# 03. 전처리

- 입력: `data/processed/seoul_bike_clean.csv` (01에서 저장)
- 하는 일: humidity 0 보정 → 운영하지 않은 시간 제외 → 날짜 변수 추출 → 범주형 인코딩 → train/validation/test 분할(70/15/15) → 표준화
- 출력: `data/processed/X_train.csv`, `X_val.csv`, `X_test.csv`, `y_train.csv`, `y_val.csv`, `y_test.csv` → 04, 05에서 사용

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트(notebooks/의 상위 폴더)를 import 경로에 추가해서 src 패키지를 사용
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
sys.path.append(str(ROOT))

import pandas as pd

from src.config import SEED, TARGET
from src.data import load_clean, save_splits

df = load_clean()
print(df.shape)

In [ ]:
import numpy as np

# humidity == 0 인 17행은 같은 행의 기온·이슬점과 맞지 않음 (측정 오류로 추정)
# → 기온·이슬점으로 상대습도를 계산(Magnus 공식)해서 대체
def humidity_from_dew_point(temp, dew_point):
    a, b = 17.625, 243.04
    return 100 * np.exp(a * dew_point / (b + dew_point)) / np.exp(a * temp / (b + temp))

zero_humidity = df['humidity'] == 0
estimated = humidity_from_dew_point(df.loc[zero_humidity, 'temperature'], df.loc[zero_humidity, 'dew_point_temp'])
df.loc[zero_humidity, 'humidity'] = estimated.round().astype(int)

print('보정한 행 수:', zero_humidity.sum())
print('보정 후 humidity == 0 행 수:', (df['humidity'] == 0).sum())
df.loc[zero_humidity, ['date', 'hour', 'temperature', 'dew_point_temp', 'humidity']]

In [ ]:
# 운영하지 않은 시간(functioning_day == 'No', 295행)은 대여 수가 전부 0 → 수요가 아니므로 제외
# 제외 후에는 값이 'Yes' 하나뿐이라 컬럼도 삭제
data = df[df['functioning_day'] == 'Yes'].drop(columns='functioning_day').reset_index(drop=True)
print(df.shape, '->', data.shape)

In [ ]:
# 날짜에서 요일(0=월 ~ 6=일), 월, 주말 여부 추출 후 date 삭제
data['day_of_week'] = data['date'].dt.dayofweek
data['month'] = data['date'].dt.month
data['is_weekend'] = (data['day_of_week'] >= 5).astype(int)
data = data.drop(columns='date')
data[['day_of_week', 'month', 'is_weekend']].head()

In [ ]:
# 범주형 인코딩: seasons는 원-핫, holiday는 값이 2개라 0/1 한 컬럼으로 변환
data = pd.get_dummies(data, columns=['seasons'], dtype=int)
data['holiday'] = (data['holiday'] == 'Holiday').astype(int)
print(data.shape)
data.dtypes

In [ ]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=TARGET)
y = data[TARGET]

# 무작위 분할: train 70 / validation 15 / test 15
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=SEED)

for name, part in [('train', X_train), ('validation', X_val), ('test', X_test)]:
    print(f'{name:10s} {len(part):5d}행 ({len(part) / len(X):.1%})')

In [ ]:
from sklearn.preprocessing import StandardScaler

# 연속형·순서형 변수만 표준화 (0/1 컬럼은 그대로)
# 데이터 누수 방지: train으로만 fit, validation/test는 transform만
binary_cols = ['holiday', 'is_weekend'] + [c for c in X.columns if c.startswith('seasons_')]
scale_cols = [c for c in X.columns if c not in binary_cols]

scaler = StandardScaler()
X_train = X_train.copy()
X_val = X_val.copy()
X_test = X_test.copy()
X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_val[scale_cols] = scaler.transform(X_val[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

print('표준화한 컬럼:', scale_cols)
print('그대로 둔 컬럼:', binary_cols)
X_train[scale_cols].agg(['mean', 'std']).round(3)

In [ ]:
# 분할·표준화된 데이터 저장 (04, 05에서 사용)
save_splits(X_train, X_val, X_test, y_train, y_val, y_test)